# SentencePiece Tokenizer

# Tokenizer
**Tokenization이란?** text를 word 또는 sub-word로 분리하는 것


**구분**
1. 사전방식 Rule-based : 특정 언어 지식이 필요
    - 공백 또는 구두점등 언어규칙에 따라 분리
    - keras tokenizer
    - konlpy tokenizer(Okt, Komoran, Mecab ...)
    - 출력 예시:
        - (코, 명사)
        - (코로나, 명사)
        - (로, 조사)
    - 문제점:
        - 너무 방대한 단어사전이 필요하고, 이에 따라 embedding matrix 또한 커진다.
        - memory/time complexity 증가
        - 한국어 형태소 사전인 NIADic은 약 93만 개의 단어를 포함
        - 영어 대표적 규칙기반 사전 WordNet 15만개 단어

2. sub-word 방식 : 언어지식 불필요, 다수의 언어에 공통으로 적용가능
    - WPM (BPE 알고리즘)
    - sentensepiece (BPE 알고리즘 + Unigram 알고리즘)
    - 출력 예시:
        - 코, 로, 나, 가, 심, 하, 다
        - 코로, 코로나, 로나...
        - `가</w>`, `다</w>`
    - 원칙
        - **빈번히 사용되는 단어는 더 작은 subword로 나뉘어 지면 안된다.**
        - **가끔 사용되는 단어는 의미있는 subword로 나뉘어 져야 한다.**
    - 교착어(한국어, 터키어, 일본어 등)의 token 화에 유용하다.
    - 예: Bert 104개 국어버젼은 110,000 단어사전을 사용.

| **Tokenizer 방식** | **토큰 단위**                      | **vocab size** | **미등록 단어에 대한 가정**                                                                                  |
|---------------------|------------------------------------|----------------|-------------------------------------------------------------------------------------------------------------|
| **사전 기반**       | 알려진 단어/형태소의 결합           | unlimited       | - 알려진 단어/형태소의 결합이라고 가정<br>- 필요한 형태소 분석 가능<br>- 사전에 등록되지 않은 단어는 UNK 처리 |
| **sub-word**        | 알려진 글자 및 sub-word            | fixed           | - 알려진 sub-words로 분해<br>- 예: appear → app + ear<br>- 자주 등장하는 단어를 제대로 인식 가능<br>- UNK의 개수 최소화 |


**Tokenizer 연도별 출현순서**


<table border="1">
  <tr>
    <th>토크나이저</th>
    <th>출현 시기</th>
    <th>설명</th>
  </tr>
  <tr>
    <td>BPE 알고리즘</td>
    <td>1990년대</td>
    <td>실제 NLP적용은 응용버젼인 WordPiece Tokenizer를 통해 2016년에 사용되었다.</td>
  </tr>
  <tr>
    <td>WordPiece Tokenizer</td>
    <td>2016년</td>
    <td>자주 사용하는 서브워드를 기반으로 한 토크나이저, BERT에서 주로 사용됨.</td>
  </tr>
  <tr>
    <td>Unigram 알고리즘</td>
    <td>2018년</td>
    <td>
        n-그램 모델 방식으로 n=1인 경우(즉, 각 단어가 이전 단어들과는 독립적으로 발생한다고 가정하고) 가장 자주 나타나는 서브워드들을 선택해 그 확률을 최대화하는 방식으로 최적의 서브워드 집합을 선택함.
        <br>
        SentencePiece의 일부로 소개됨.
    </td>
  </tr>
  <tr>
    <td>SentencePiece Tokenizer</td>
    <td>2018년</td>
    <td>BPE와 Unigram 모델을 지원하며, 공백이 없는 텍스트 처리에 유용한 토크나이저.</td>
  </tr>
  <tr>
    <td>Subword Text Encoder</td>
    <td>2018년</td>
    <td>TensorFlow의 서브워드 기반 토크나이저로 효율적인 텍스트 처리를 지원.</td>
  </tr>
  <tr>
    <td>BertWordPieceTokenizer</td>
    <td>2018년</td>
    <td>특히 BERT에서 사용되는 WordPiece 토크나이저.</td>
  </tr>
  <tr>
    <td>CharBPETokenizer (문자 수준 Byte Pair Encoding)</td>
    <td>2019년</td>
    <td>Hugging Face의 tokenizers 라이브러리에 추가</td>
  </tr>
  <tr>
    <td>ByteLevelBPETokenizer</td>
    <td>2019년</td>
    <td>바이트 수준의 BPE 알고리즘을 사용, GPT-2에서 주로 사용됨.</td>
  </tr>
  <tr>
    <td>SentencePieceBPETokenizer</td>
    <td>2020년</td>
    <td>SentencePiece와 BPE 방식을 결합한 효율적인 토크나이저.</td>
  </tr>

</table>



## BPE(Byte Pair Encoding)

BPE(Byte Pair Encoding) 알고리즘은 1994년에 제안된 데이터 압축 방법이다. 자연어 처리에서는 서브워드 분리 알고리즘으로 응용된다. 기본 원리는 연속적으로 가장 많이 등장하는 글자(바이트) 쌍을 찾아 하나의 글자로 병합하는 것이다.

예를 들어, 문자열

```
aaabdaaabac
```

1. 가장 많이 등장하는 쌍 'aa'를 'Z'로 치환

```
ZabdZabac
Z=aa
```

2. 다음으로 많이 등장하는 쌍 'ab'를 'Y'로 치환

```
ZYdZYac
Y=ab
Z=aa
```

3. 다음으로 많이 등장하는 쌍 'ZY'를 'X'로 치환

```
XdXac
X=ZY
Y=ab
Z=aa
```

더 이상 병합할 쌍이 없으므로 BPE는 여기서 종료된다.

## Google SentencePiece Tokenizer

- 논문 : https://arxiv.org/pdf/1808.06226.pdf
- 센텐스피스 깃허브 : https://github.com/google/sentencepiece

구글은 BPE 알고리즘과 Unigram Language Model Tokenizer를 구현한 센텐스피스를 깃허브에 공개하였다.

subword 분리 알고리즘을 사용하기 위해서는 데이터에 단어 토큰화를 먼저 진행한 상태여야 하는데, 이 subword 알고리즘을 모든 언어에 사용하는 것은 쉽지 않다.

영어와 달리 한국어와 같은 언어는 단어 토큰화부터가 쉽지 않기 때문이다.

그런데, 이런 사전 토큰화 작업인 `pretokenization` 없이 전처리를 하지 않은 데이터(`raw data`)에 바로 subword 토크나이저를 사용할 수 있다면, 이 토크나이저는 그 어떤 언어에도 적용할 수 있는 토크나이저가 될 것이다.

`sentencepiece`는 이 이점을 살려서 구현되었다. 센텐스피스는 사전 토큰화 작업 없이 단어 분리 토큰화를 수행하므로 언어에 종속되지 않는다.

`rule-based tokenizer`인 `okt` 같은 경우 미리 학습된 단어사전이 존재한다. `sentencepiece`와 같은 `subword` 방식은 미리 준비된 `vocab`이 없고 데이터로 학습 과정이 필요하다.

Google SentencePiece는 언어 및 스크립트에 독립적인 서브워드 토크나이저로, 텍스트 전처리와 토크나이징을 위해 설계된 도구이다. 다양한 자연어 처리(NLP) 모델에서 사용 가능하며, 특정 프레임워크에 종속되지 않는다.

### 주요 특징

1. 독립적 설계

   - 기존의 토크나이저는 공백 기반으로 동작하지만, SentencePiece는 문장에서 공백을 그대로 하나의 문자로 간주하여 처리한다.
   - 유니코드 및 비유니코드 언어 모두 지원하며, 언어 구조와 관계없이 사용할 수 있다.

2. 학습 방식

   - BPE(Byte Pair Encoding): 빈도가 높은 문자 쌍을 병합하며 서브워드를 생성.
   - Unigram Language Model: 가장 적합한 서브워드 세트를 선택하여 전체 문자열의 확률을 최대화.

3. 유연한 토큰화

   - 텍스트 전처리(소문자 변환, 정규화) 없이 원본 데이터를 그대로 처리 가능.
   - 특수문자, 공백 포함한 서브워드를 그대로 학습 및 생성.

4. 사용성

   - 텍스트 파일 형식으로 데이터를 입력받아 어휘 집합을 생성하며, 다양한 출력 형식을 제공.
   - TensorFlow, PyTorch 등 다양한 프레임워크와 통합 가능.

### 활용 사례

- 언어 모델링: BERT, T5 등 서브워드 기반 NLP 모델에 활용.
- 다국어 지원: 비유니코드 및 다국어 환경의 NLP 작업에서 필수적인 도구.
- 텍스트 생성 및 번역: 다양한 형태의 언어 데이터 전처리 및 토크나이징.

## 데이터 로드

In [1]:
import json
from pathlib import Path

data_path = Path('data/1-1.여성의류(114).json')

with open(data_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(type(data))
print(len(data))
print(data[0].keys())

<class 'list'>
100
dict_keys(['Index', 'RawText', 'Source', 'Domain', 'MainCategory', 'ProductName', 'ReviewScore', 'Syllable', 'Word', 'RDate', 'GeneralPolarity', 'Aspects'])


In [2]:
# 첫 번째 리뷰 데이터 확인
data[0]

{'Index': '47192',
 'RawText': '얇고 가벼워요 편한데 색이나 모양도 예쁘고요 얇은 밴딩 바지들은 앞이 좀 민망할 때가 있는데 포켓이 있어 앞 라인도 안 민망하고 숨겨진 밴딩때문에 정말 편합니다 색감도 예쁘고 깔끔해서 좋아요 시원함과 편안함 때문에 아웃도어룩을 많이 입었는데 이번 여름은 진 세 장 번갈아 입으며 살고 있어요. ',
 'Source': '쇼핑몰',
 'Domain': '패션',
 'MainCategory': '여성의류',
 'ProductName': 'OO 여성 울** 쿨썸머 데님 3종 ',
 'ReviewScore': '100',
 'Syllable': '164',
 'Word': '43',
 'RDate': '20190712',
 'GeneralPolarity': '1',
 'Aspects': [{'Aspect': '두께',
   'SentimentText': '얇고 ',
   'SentimentWord': '1',
   'SentimentPolarity': '1'},
  {'Aspect': '무게',
   'SentimentText': '가벼워요',
   'SentimentWord': '1',
   'SentimentPolarity': '1'},
  {'Aspect': '착용감',
   'SentimentText': '편한데',
   'SentimentWord': '1',
   'SentimentPolarity': '1'},
  {'Aspect': '색상',
   'SentimentText': '색이나 모양도 예쁘고요',
   'SentimentWord': '3',
   'SentimentPolarity': '1'},
  {'Aspect': '디자인',
   'SentimentText': '모양도 예쁘고요',
   'SentimentWord': '2',
   'SentimentPolarity': '1'},
  {'Aspect': '디자인',
   'SentimentText': ' 포켓이 있어 앞 라인도 안 민망하고',

In [3]:
# 리뷰 원문만 추출
reviews = [item['RawText'] for item in data if item.get('RawText')]

print(len(reviews))
print(reviews[:3])

100
['얇고 가벼워요 편한데 색이나 모양도 예쁘고요 얇은 밴딩 바지들은 앞이 좀 민망할 때가 있는데 포켓이 있어 앞 라인도 안 민망하고 숨겨진 밴딩때문에 정말 편합니다 색감도 예쁘고 깔끔해서 좋아요 시원함과 편안함 때문에 아웃도어룩을 많이 입었는데 이번 여름은 진 세 장 번갈아 입으며 살고 있어요. ', '가격대비상품괜찮네요. 착용감도편안하구요. 잘입겠습니다.', '몇년 전 OOO 쿨데님 사셨는데 이번에 새로 샀습니다. 두가지 색상이 비슷한게 아쉽지만 소재가 마음에 든다고 하네요. 얇고 스판끼도 많아서 편하게 입을 스타일이래요.']


## SentencePiece 학습용 코퍼스 파일 생성
SentencePiece는 텍스트 파일을 입력으로 받아 tokenizer model을 학습한다.
따라서 리뷰가 담긴 텍스트 파일을 생성한다.

In [4]:
corpus_path = Path('data/female_fashion_reviews.txt')

with open(corpus_path, 'w', encoding='utf-8') as f:
    for review in reviews:
        review = review.replace('\n', ' ').strip()
        if review:
            f.write(review + '\n')

print(corpus_path.exists())

True


## SentencePiece 설치

In [5]:
%pip install sentencepiece -q

Note: you may need to restart the kernel to use updated packages.


## SentencePiece 모델 학습

In [ ]:
import sentencepiece as spm 

input_file = 'data/female_fashion_reviews.txt'
prefix = 'model/female_fashion'
vocab_size = 1000

spm.SentencePieceTrainer.Train(
    input=input_file,               # 학습할 데이터
    model_prefix=prefix,            # 생성 될 모델 저장 경로
    vocab_size=vocab_size,          # vocalbulary 크기
    model_type='bpe',               # subword 학습 방식
    character_coverage=0.9995,      # 학습 데이터 문자 중 99.95% 커버할 수 있는 기본 문자 포함
    pad_id=0,                       
    unk_id=1,
    bos_id=2,
    eos_id=3,
    hard_vocab_limit=False          # 데이터 여건 상 vocab_size를 맞추지 못해도 학습 진행
)

In [9]:
# 학습 결과 파일 확인
import os
print(os.path.exists(f'{prefix}.model'))
print(os.path.exists(f'{prefix}.vocab'))

True
True


## SentencePieceProcessor 로드
학습 된 `.model` 파일을 `SentencePieceProcessor` 로 로드하면 tokenizer 처럼 사용할 수 있다.

In [10]:
sp = spm.SentencePieceProcessor()
sp.Load(f'{prefix}.model')

print('vocab_size : ', sp.get_piece_size())
print('pad id : ', sp.pad_id())
print('unk id : ', sp.unk_id())
print('bos id : ', sp.bos_id())
print('eos id : ', sp.eos_id())

vocab_size :  1000
pad id :  0
unk id :  1
bos id :  2
eos id :  3


## 토큰화 결과 확인

In [11]:
# 리뷰 데이터 토큰화 확인
for review in reviews[:3]:
    print(f'Test : {review}')
    print(f'토큰 : {sp.encode_as_pieces(review)}')          # subword 토큰 목록 반환
    print(f'정수인코딩 : {sp.encode_as_ids(review)}')        # 정수 id 목록 반환
    print()

Test : 얇고 가벼워요 편한데 색이나 모양도 예쁘고요 얇은 밴딩 바지들은 앞이 좀 민망할 때가 있는데 포켓이 있어 앞 라인도 안 민망하고 숨겨진 밴딩때문에 정말 편합니다 색감도 예쁘고 깔끔해서 좋아요 시원함과 편안함 때문에 아웃도어룩을 많이 입었는데 이번 여름은 진 세 장 번갈아 입으며 살고 있어요. 
토큰 : ['▁얇고', '▁가', '벼', '워요', '▁편', '한데', '▁색', '이나', '▁모', '양도', '▁예쁘고', '요', '▁얇', '은', '▁밴딩', '▁바지', '들은', '▁앞', '이', '▁좀', '▁민망', '할', '▁때', '가', '▁있', '는데', '▁포', '켓', '이', '▁있어', '▁앞', '▁라', '인', '도', '▁안', '▁민망', '하고', '▁', '숨', '겨', '진', '▁밴딩', '때', '문에', '▁정말', '▁편합니다', '▁색', '감도', '▁예쁘고', '▁', '깔', '끔', '해서', '▁좋아요', '▁시원', '함', '과', '▁편안', '함', '▁때', '문에', '▁아', '웃', '도', '어', '룩', '을', '▁많이', '▁입', '었는데', '▁이번', '▁여름', '은', '▁진', '▁세', '▁장', '▁번', '갈아', '▁입으', '며', '▁살', '고', '▁있어요', '.']
정수인코딩 : [252, 16, 931, 458, 12, 158, 47, 199, 80, 452, 173, 513, 68, 533, 323, 23, 239, 285, 514, 36, 498, 592, 375, 517, 20, 13, 290, 979, 514, 137, 285, 377, 543, 524, 50, 498, 19, 512, 945, 790, 587, 323, 723, 446, 87, 364, 47, 291, 173, 512, 696, 884, 43, 31, 135, 680, 682, 211, 680, 375, 446, 32, 835, 524, 523, 911,